In [21]:
import pandas as pd

sentiment = pd.read_csv("daily_sentiment.csv")
prices = pd.read_csv("../WEEK1/stock_data.csv")


In [22]:
# Reset index
prices = prices.reset_index()

# If index column is not called Date, rename it
if "Date" not in prices.columns:
    prices = prices.rename(columns={prices.columns[0]: "Date"})

prices["Date"] = pd.to_datetime(prices["Date"]).dt.date
sentiment["date"] = pd.to_datetime(sentiment["date"]).dt.date



In [23]:
data = prices.merge(
    sentiment,
    left_on="Date",
    right_on="date",
    how="inner"
)

data.head()


,Date,Price,Close,High,Low,Open,Volume,date,sentiment_score


In [24]:
print(prices.columns)
print(sentiment.columns)

import numpy as np

# Log returns
data["log_return"] = np.log(data["Close"] / data["Close"].shift(1))

# Rolling volatility (10-day)
data["volatility"] = data["log_return"].rolling(10).std()

# RSI (14-day)
delta = data["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

rs = gain.rolling(14).mean() / loss.rolling(14).mean()
data["RSI"] = 100 - (100 / (1 + rs))

# Clean NaNs from rolling calculations
data = data.dropna().reset_index(drop=True)

data.head()


Index(['Date', 'Price', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
Index(['date', 'sentiment_score'], dtype='object')


,Date,Price,Close,High,Low,Open,Volume,date,sentiment_score,log_return,volatility,RSI


In [25]:
features = data[
    ["log_return", "volatility", "RSI", "Volume", "sentiment_score"]
]

target = data["log_return"].shift(-1)

# Remove NaNs from shifting
features = features.iloc[:-1]
target = target.iloc[:-1]

features.head(), target.head()



(Empty DataFrame
 Columns: [log_return, volatility, RSI, Volume, sentiment_score]
 Index: [],
 Series([], Name: log_return, dtype: object))